In [30]:
globals().clear()

In [31]:
from pathlib import Path
import cv2
import numpy as np

IMG_SIZE = 128
BASE_DIR = Path("data/kaggle_3m")

mask_paths = list(BASE_DIR.rglob("*mask*.*"))
image_paths = [Path(str(m).replace("_mask", "")) for m in mask_paths]

print(f" {len(mask_paths)} pairs of picture-mask")

def process_image(path):
    img = cv2.imread(str(path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return cv2.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0

def process_mask(path):
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    return cv2.resize(mask, (IMG_SIZE, IMG_SIZE)) / 255.0

#CNN-friendly dims
X = np.array([process_image(p) for p in image_paths], dtype=np.float32)
Y = np.array([process_mask(p) for p in mask_paths], dtype=np.float32)

Y = Y[..., np.newaxis]

print("X shape:", X.shape)
print("Y shape:", Y.shape)

 3929 pairs of picture-mask
X shape: (3929, 128, 128, 3)
Y shape: (3929, 128, 128, 1)


In [32]:

has_tumor = np.sum(Y, axis=(1, 2, 3)) > 0

total_images = len(Y)
positive_cases = np.sum(has_tumor)
negative_cases = total_images - positive_cases

print(f"Total: {total_images}")
print(f"Cancer: {positive_cases} ({positive_cases/total_images*100:.2f}%)")
print(f"No Cancer: {negative_cases} ({negative_cases/total_images*100:.2f}%)")

Total: 3929
Cancer: 1373 (34.95%)
No Cancer: 2556 (65.05%)


In [33]:
import keras
model=keras.Sequential()
model.add(keras.layers.Conv2D(input_shape=(128,128,3),
                              filters=32,
                              kernel_size=(3,3)))
model.add(keras.layers.MaxPool2D(pool_size=(2,2)))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(1,activation='sigmoid'))

model.compile(optimizer=keras.optimizers.Adam(),
              loss=keras.losses.binary_crossentropy,
              metrics=['accuracy'])

C:\Users\Antonis\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [34]:
Y_class = (np.sum(Y, axis=(1, 2, 3)) > 0).astype(np.float32)

print("New shape Y_class:", Y_class.shape) 


New shape Y_class: (3929,)


In [35]:
model.fit(X,Y_class,batch_size=32,epochs=3, verbose=2)

Epoch 1/3
123/123 - 12s - 95ms/step - accuracy: 0.7546 - loss: 0.5124
Epoch 2/3
123/123 - 9s - 76ms/step - accuracy: 0.8292 - loss: 0.3663
Epoch 3/3
123/123 - 9s - 76ms/step - accuracy: 0.8638 - loss: 0.3127


KeyError: '_oh'